# experiment2

## Reproducibility bootstrap
Run first. Resolves paths for the authors' Drive, a fresh Colab (clones the anon repo), or a local clone. No edits needed.

In [ ]:
# === Reproducibility bootstrap (public bundle) ===
# Resolves all paths for: (a) authors' Google Drive, (b) fresh Colab (clones repo),
# (c) local clone. Sets CODE_DIR, DATA_DIR, RESULTS_DIR, DATA_PATH. Run first; no edits needed.
import os, sys
from pathlib import Path

RESULTS_SUBFOLDER = "experiment2"
DATA_FILENAME = None   # None for synthetic experiments

def _resolve():
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dr = Path('/content/drive/MyDrive')
        if (dr/'GNAVAR'/'code'/'gnavar_core.py').exists():
            b = dr/'GNAVAR'; return b/'code', b/'data', b/'results'
        # Fresh Colab without the authors' Drive: the repo files must be present in the
        # session. Anonymous-review repos cannot be git-cloned, so upload the bundle:
        #   1) Download the ZIP from the Anonymous GitHub page (Download / ZIP button).
        #   2) In Colab, upload the ZIP via the Files pane, then in a cell run:
        #        !unzip -o your_bundle.zip
        #   3) %cd into the unzipped repo folder, then run this notebook.
        for cand in [Path('/content')/'ICDM-GNAVAR-EDAE', Path.cwd()]:
            if (cand/'src'/'gnavar_core.py').exists():
                return cand/'src', cand/'data', cand/'results'
        raise FileNotFoundError(
            'Repo files not found in the Colab session. Download the ZIP from the '
            'Anonymous GitHub page, upload and unzip it here, then %cd into the folder '
            'and re-run. See the repository README, Path B, Option B1.')
    except ImportError:
        repo = Path.cwd()
        while repo != repo.parent and not (repo/'verify_paper_numbers.py').exists():
            repo = repo.parent
        return repo/'src', repo/'data', repo/'results'

CODE_DIR, DATA_DIR, RESULTS_ROOT = _resolve()
sys.path.insert(0, str(CODE_DIR))
RESULTS_DIR = RESULTS_ROOT / RESULTS_SUBFOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = (DATA_DIR / DATA_FILENAME) if DATA_FILENAME else None
DRIVE_ROOT = str(CODE_DIR.parent.parent)  # back-compat for any cell referencing DRIVE_ROOT
print('CODE_DIR    =', CODE_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
if DATA_PATH: print('DATA_PATH   =', DATA_PATH)
import numpy as np
import pandas as pd
import json, hashlib, datetime, time, itertools, platform
try:
    import torch
    import torch.nn as nn
except Exception:
    pass
from gnavar_core import *  # model, generator, fit/eval utils
# === end bootstrap ===


# Experiment 2: Support-Collapse Phase Transition

**Purpose.** Empirically validate Corollary 8.2 of the G-NAVAR identifiability
paper. As the support of jointly-modulating variables collapses onto a
lower-dimensional manifold, the identifiability of multiplicative gate
structure fails in a precise, predictable way.

## Setup

Same DGP as Experiment 1, but with a controlled coupling between modulators $x_3$ and $x_5$:
$$x_{5,t} = \rho \cdot x_{3,t} + \sqrt{1-\rho^2} \cdot \eta_{5,t}$$
(applied to the AR(1) innovations, so $x_5$ remains stationary).

- At $\rho = 0$: $x_3$ and $x_5$ are independent; theorem applies; modulator-set recovery should be perfect (replicating Experiment 1).
- At $\rho = 1$: $x_5 = x_3$ on support; corollary applies; modulator-set recovery degrades to chance.

## Three measured quantities per $\rho$

1. **Modulator-set recovery accuracy** — fraction of trials recovering true modulator sets exactly.
2. **Gate $L^2$ error** — distance from fitted gates to true gates (gauge-normalized).
3. **Identifiability gap** — $L^2$ distance between gates from two independently-initialized fits on the *same* data. At $\rho = 0$ this should be near zero (gates identifiable); at $\rho = 1$ it grows (different fits find different equivalent solutions).

## Outputs (under `/content/drive/MyDrive/GNAVAR/results/experiment2/`)

- `results.csv`: per-trial metrics, incremental.
- `phase_transition.png`: headline figure — three panels as a function of $\rho$.
- `summary.txt`: aggregated by $\rho$.


## Cell 1: Drive mount and module path

## Cell 2: Imports from gnavar_core

In [ ]:
from gnavar_core import *
# Brings in: Config, true_f/g functions, simulate_dgp, simulate_dgp_coupled,
# make_lag_tensor, BatchedMLP, GNAVAR, fit_gnavar, all metrics, save/load,
# effective_rank, effective_rank_subset, DEVICE, USE_AMP.

import time, json, hashlib, platform, datetime
from dataclasses import asdict
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print(f'Device: {DEVICE} | Mixed precision: {USE_AMP}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')


## Cell 3: Experiment 2 configuration

Same Config as Experiment 1, plus the $\rho$ sweep grid. Uses the
tuned `l1_lambda=0.005` from Experiment 1.

In [ ]:
# Base config
cfg = Config(
    sample_sizes=(25000,),
    n_seeds=10,                # bumped from 5 -> 10 for tighter error bars at ICDM
    base_seed=42,
    n_epochs=300,
    l1_lambda=0.005,            # tuned in Experiment 1
    triviality_threshold=0.001,
)

# Rho sweep: dense near rho=1 where the phase transition happens
RHO_VALUES = (0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0)

# T for Experiment 2: fixed at 25k (Experiment 1 showed clean recovery here)
T_EXP2 = 25000

# Restart-and-keep-best: addresses optimizer bimodality at rho=0.
# Control study v2 verdict should determine whether this is sufficient.
N_RESTARTS = 3

print(json.dumps(asdict(cfg), indent=2))
print(f'\nRho values: {RHO_VALUES}')
print(f'T (fixed):  {T_EXP2}')
print(f'Replications per rho: {cfg.n_seeds}')
print(f'Total trials: {len(RHO_VALUES) * cfg.n_seeds} '
      f'(each with 2 independent fits, each best of {N_RESTARTS} restarts)')


## Cell 4: Identifiability-gap metric

Train two G-NAVAR models with different random seeds on the *same* simulated
data, then measure the $L^2$ distance between their gate functions over the
empirical support. Under identifiability, both fits converge to the same
gates (up to gauge); under non-identifiability, they can find different
equivalent solutions and the distance grows.


In [ ]:
@torch.no_grad()
def gate_pair_distance(model_a: 'GNAVAR', model_b: 'GNAVAR',
                       X_lag_t: torch.Tensor, j: int, k: int) -> float:
    """
    Gauge-normalized L2 distance between gate g_{j,k} from two models,
    evaluated on the empirical lag-block samples of x_k.
    """
    x_block = X_lag_t[:, k, :]
    g_a = model_a.evaluate_gate(j=j, k=k, x_block=x_block).cpu().numpy()
    g_b = model_b.evaluate_gate(j=j, k=k, x_block=x_block).cpu().numpy()
    g_a_n, _ = gauge_normalize(g_a)
    g_b_n, _ = gauge_normalize(g_b)
    return float(np.sqrt(np.mean((g_a_n - g_b_n) ** 2)))

## Cell 5: Trial driver

In [ ]:
def run_trial_exp2(rho: float, seed: int, cfg: Config, T: int,
                   verbose: bool = False) -> dict:
    """
    One trial: simulate at given rho, fit two models on same data, evaluate
    recovery metrics and identifiability gap.

    We report recovery metrics from BOTH models (model_a and model_b),
    averaged. Each model's metrics are also stored separately so a reader
    can verify that the per-fit results are consistent.
    """
    t0 = time.time()
    X = simulate_dgp_coupled(T=T, cfg=cfg, seed=seed, rho=rho)

    # Two independent fits on the SAME data (different init seeds)
    model_a = fit_gnavar_with_restarts(X, cfg, seed=seed + 1000, n_restarts=N_RESTARTS, verbose=False)
    model_b = fit_gnavar_with_restarts(X, cfg, seed=seed + 2000, n_restarts=N_RESTARTS, verbose=False)
    model_a.eval(); model_b.eval()

    X_lag_np, _ = make_lag_tensor(X, cfg.K)
    X_lag_t = torch.from_numpy(X_lag_np).to(DEVICE)

    # --- Modulator-set recovery (each model, then "either-correct" and "both-correct") ---
    true_M = {0: {1}, 2: {3}}
    detected_a = {
        0: detect_modulator_set(model_a, X_lag_t, j=0, threshold=cfg.triviality_threshold),
        2: detect_modulator_set(model_a, X_lag_t, j=2, threshold=cfg.triviality_threshold),
    }
    detected_b = {
        0: detect_modulator_set(model_b, X_lag_t, j=0, threshold=cfg.triviality_threshold),
        2: detect_modulator_set(model_b, X_lag_t, j=2, threshold=cfg.triviality_threshold),
    }
    correct_a = int(detected_a[0] == true_M[0] and detected_a[2] == true_M[2])
    correct_b = int(detected_b[0] == true_M[0] and detected_b[2] == true_M[2])

    # --- Gate L^2 error: each model, then average ---
    err_g123_a = gate_l2_error(model_a, X_lag_t, j=0, k=1, true_gate_fn=true_g123)
    err_g145_a = gate_l2_error(model_a, X_lag_t, j=2, k=3, true_gate_fn=true_g145)
    err_g123_b = gate_l2_error(model_b, X_lag_t, j=0, k=1, true_gate_fn=true_g123)
    err_g145_b = gate_l2_error(model_b, X_lag_t, j=2, k=3, true_gate_fn=true_g145)
    err_g123 = 0.5 * (err_g123_a + err_g123_b)
    err_g145 = 0.5 * (err_g145_a + err_g145_b)

    # --- Identifiability gap: distance between the two fits ---
    gap_g123 = gate_pair_distance(model_a, model_b, X_lag_t, j=0, k=1)
    gap_g145 = gate_pair_distance(model_a, model_b, X_lag_t, j=2, k=3)

    # --- Support-richness diagnostic on the modulator pair (x3, x5) ---
    r_eff_x3_x5 = effective_rank_subset(X_lag_np, subset_indices=[1, 3])

    elapsed = time.time() - t0
    return {
        'rho': rho,
        'seed': seed,
        'T': T,
        # Modulator-set recovery: report each model and the "both correct" measure
        'modulator_correct_a': correct_a,
        'modulator_correct_b': correct_b,
        'modulator_correct_both': int(correct_a and correct_b),
        'M_source0_a': sorted(detected_a[0]),
        'M_source2_a': sorted(detected_a[2]),
        'M_source0_b': sorted(detected_b[0]),
        'M_source2_b': sorted(detected_b[2]),
        # Gate L^2: per-model and average
        'err_g123_a': err_g123_a,
        'err_g123_b': err_g123_b,
        'err_g145_a': err_g145_a,
        'err_g145_b': err_g145_b,
        'err_g123': err_g123,   # average
        'err_g145': err_g145,   # average
        # Identifiability gap (the headline metric)
        'gap_g123': gap_g123,
        'gap_g145': gap_g145,
        # Diagnostic
        'r_eff_x3_x5': r_eff_x3_x5,
        'elapsed_seconds': elapsed,
    }


## Cell 6: Resume-aware experiment driver

In [ ]:
RESULTS_CSV = RESULTS_DIR / 'results.csv'

def load_existing_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        completed = set(zip(df['rho'].astype(float), df['seed'].astype(int)))
        return df, completed
    return pd.DataFrame(), set()

def append_result(row: dict):
    write_header = not RESULTS_CSV.exists()
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)

def _flatten_sets(d: dict) -> dict:
    """Convert set-valued fields to strings for CSV compatibility."""
    out = dict(d)
    for k in ['M_source0_a', 'M_source2_a', 'M_source0_b', 'M_source2_b']:
        if k in out:
            out[k] = str(out[k])
    return out

def run_experiment_2(cfg: Config, rho_values=RHO_VALUES, T=T_EXP2):
    _, completed = load_existing_results()
    print(f'Already completed: {len(completed)} trials')
    plan = [(rho, cfg.base_seed + 1000 * s)
            for rho in rho_values for s in range(cfg.n_seeds)]
    todo = [(r, sd) for (r, sd) in plan if (r, sd) not in completed]
    print(f'To run: {len(todo)} trials')
    for i, (rho, seed) in enumerate(todo, 1):
        print(f'\n[{i}/{len(todo)}] rho={rho}, seed={seed}', flush=True)
        result = run_trial_exp2(rho=rho, seed=seed, cfg=cfg, T=T, verbose=False)
        append_result(_flatten_sets(result))
        print(f'  done in {result["elapsed_seconds"]:.1f}s | '
              f'correct (a/b/both)={result["modulator_correct_a"]}/{result["modulator_correct_b"]}/{result["modulator_correct_both"]} | '
              f'gap_g123={result["gap_g123"]:.3f}, gap_g145={result["gap_g145"]:.3f} | '
              f'r_eff={result["r_eff_x3_x5"]:.2f}', flush=True)
    return load_existing_results()[0]


## Cell 7: Phase transition figure (headline visual)

In [ ]:
def plot_phase_transition(df: pd.DataFrame):
    """
    Three-panel headline figure: modulator accuracy, gate L2 error, identifiability gap,
    each as a function of rho.

    Importantly, we plot the rho=0 value of each metric as a horizontal reference line
    on the relevant panels. This makes the phase transition a RELATIVE statement
    (recovery degrades from the identifiable-regime baseline) rather than an absolute
    one (recovery falls to chance), which is what the data actually supports.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    # Reference baselines from rho = 0
    df_baseline = df[df['rho'] == df['rho'].min()]
    base_modulator = df_baseline['modulator_correct_both'].mean()
    base_err = ((df_baseline['err_g123'] + df_baseline['err_g145']) / 2).mean()
    base_gap = ((df_baseline['gap_g123'] + df_baseline['gap_g145']) / 2).mean()

    # Panel 1: modulator-set recovery accuracy (both fits agreeing)
    g = df.groupby('rho')['modulator_correct_both'].agg(['mean', 'std'])
    axes[0].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C0',
                     label='both fits recover M')
    axes[0].axhline(base_modulator, color='gray', linestyle=':', alpha=0.6,
                    label=f'rho=0 baseline ({base_modulator:.2f})')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_xlabel(r'Coupling strength $\rho$')
    axes[0].set_ylabel('Modulator-set recovery')
    axes[0].set_title(r'Modulator-set recovery vs $\rho$')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc='best')

    # Panel 2: gate L2 errors (averaged across both fits, both gates)
    df2 = df.copy()
    df2['err_g_avg'] = (df2['err_g123'] + df2['err_g145']) / 2
    g = df2.groupby('rho')['err_g_avg'].agg(['mean', 'std'])
    axes[1].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C1', label='avg over gates and fits')
    axes[1].axhline(base_err, color='gray', linestyle=':', alpha=0.6,
                    label=f'rho=0 baseline ({base_err:.3f})')
    axes[1].set_xlabel(r'Coupling strength $\rho$')
    axes[1].set_ylabel(r'Gate $L^2$ error (gauge-normalized)')
    axes[1].set_title(r'Gate recovery error vs $\rho$')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='best')

    # Panel 3: identifiability gap (the deepest empirical claim).
    # The rho=0 horizontal line is the "optimization floor" — what the gap looks
    # like when two random-init fits converge to the same solution under
    # full-support identifiability. Any rise above this line indicates that
    # fits are finding distinct equivalent solutions, consistent with the
    # support-collapse non-identifiability of Corollary 8.2.
    df2['gap_avg'] = (df2['gap_g123'] + df2['gap_g145']) / 2
    g = df2.groupby('rho')['gap_avg'].agg(['mean', 'std'])
    axes[2].errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
                     marker='o', capsize=4, color='C2', label='avg across gates')
    axes[2].axhline(base_gap, color='gray', linestyle=':', alpha=0.6,
                    label=f'rho=0 baseline ({base_gap:.3f})')
    axes[2].set_xlabel(r'Coupling strength $\rho$')
    axes[2].set_ylabel(r'Identifiability gap (gate $L^2$ between two fits)')
    axes[2].set_title(r'Identifiability gap vs $\rho$')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc='best')

    fig.suptitle('Experiment 2: support-collapse phase transition (relative to rho=0 baseline)', y=1.02)
    fig.tight_layout()
    out = RESULTS_DIR / 'phase_transition.png'
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')

def plot_eff_rank_vs_recovery(df: pd.DataFrame):
    """
    Secondary figure: effective rank vs gate recovery error.
    Shows that the support-richness diagnostic predicts recovery.
    """
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    df2 = df.copy()
    df2['err_g_avg'] = (df2['err_g123'] + df2['err_g145']) / 2
    sc = ax.scatter(df2['r_eff_x3_x5'], df2['err_g_avg'],
                    c=df2['rho'], cmap='viridis', s=40)
    ax.set_xlabel(r'Effective rank $r_{\mathrm{eff}}(\{x_3, x_5\})$ (max $= 2K = 4$)')
    ax.set_ylabel(r'Gate $L^2$ error (gauge-normalized, avg)')
    ax.set_title('Support richness predicts gate recovery')
    plt.colorbar(sc, ax=ax, label=r'$\rho$')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out = RESULTS_DIR / 'eff_rank_vs_recovery.png'
    fig.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')


## Cell 8: Summary writer

In [ ]:
def write_summary_exp2(df: pd.DataFrame):
    lines = [
        'Experiment 2: Support-collapse phase transition',
        '=' * 60,
        f'Total trials: {len(df)}',
        f'T (fixed): {T_EXP2}',
        f'Replications per rho: {cfg.n_seeds}',
        '',
        'Aggregated by rho (mean +/- std):',
        '',
    ]
    for rho in sorted(df['rho'].unique()):
        sub = df[df['rho'] == rho]
        lines.append(f'rho = {rho}:')
        lines.append(f'  modulator correct (fit a):    {sub["modulator_correct_a"].mean():.3f} +/- {sub["modulator_correct_a"].std():.3f}')
        lines.append(f'  modulator correct (fit b):    {sub["modulator_correct_b"].mean():.3f} +/- {sub["modulator_correct_b"].std():.3f}')
        lines.append(f'  modulator correct (both):     {sub["modulator_correct_both"].mean():.3f} +/- {sub["modulator_correct_both"].std():.3f}')
        lines.append(f'  err_g123 (avg over a, b):     {sub["err_g123"].mean():.4f} +/- {sub["err_g123"].std():.4f}')
        lines.append(f'  err_g145 (avg over a, b):     {sub["err_g145"].mean():.4f} +/- {sub["err_g145"].std():.4f}')
        lines.append(f'  gap_g123 (a vs b):            {sub["gap_g123"].mean():.4f} +/- {sub["gap_g123"].std():.4f}')
        lines.append(f'  gap_g145 (a vs b):            {sub["gap_g145"].mean():.4f} +/- {sub["gap_g145"].std():.4f}')
        lines.append(f'  r_eff(x3,x5):                 {sub["r_eff_x3_x5"].mean():.3f} +/- {sub["r_eff_x3_x5"].std():.3f}')
        lines.append(f'  mean wall time:               {sub["elapsed_seconds"].mean():.1f}s')
        lines.append('')
    summary = '\n'.join(lines)
    (RESULTS_DIR / 'summary.txt').write_text(summary)
    print(summary)
    print(f'\nSaved {RESULTS_DIR / "summary.txt"}')


## Cell 8b: Reproducibility metadata

Writes a `metadata.json` next to `results.csv` capturing:
- Config used (full `cfg` dict)
- Sha256 of the gnavar_core.py being imported
- PyTorch / NumPy / Python versions, device, GPU name if any
- Timestamp

This satisfies the ICDM reproducibility checklist and provides receipts if
someone wants to replicate. Re-running overwrites the metadata file (latest run).

In [ ]:
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def write_reproducibility_metadata(cfg, results_dir):
    import gnavar_core as _gc
    md = {
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
        'cfg': asdict(cfg),
        'rho_values': list(RHO_VALUES),
        'T_exp2': T_EXP2,
        'n_restarts': N_RESTARTS,
        'gnavar_core_path': _gc.__file__,
        'gnavar_core_sha256': _sha256(_gc.__file__),
        'python_version': platform.python_version(),
        'torch_version': torch.__version__,
        'numpy_version': np.__version__,
        'pandas_version': pd.__version__,
        'device': str(DEVICE),
        'use_amp': USE_AMP,
        'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    out = results_dir / 'metadata.json'
    out.write_text(json.dumps(md, indent=2))
    print(f'Wrote {out}')
    return md

_md = write_reproducibility_metadata(cfg, RESULTS_DIR)
print(json.dumps(_md, indent=2))

## Cell 9: Run

Estimated wall clock on Colab T4: 9 rho values × 5 seeds × 2 fits/trial,
each fit taking ~10-15s at T=25000. Total ≈ 25-40 minutes. Resume-safe.

In [ ]:
df = run_experiment_2(cfg)
print(f'\nDone. Total rows: {len(df)}')

In [ ]:
plot_phase_transition(df)

In [ ]:
plot_eff_rank_vs_recovery(df)

In [ ]:
write_summary_exp2(df)